In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
milk_production_df = pd.read_csv("milk_production.csv")
honey_production_df = pd.read_csv("honey_production.csv")
yogurt_production_df = pd.read_csv("yogurt_production.csv")
cheese_production_df = pd.read_csv("cheese_production.csv")
coffee_production_df = pd.read_csv("coffee_production.csv")
egg_production_df = pd.read_csv("egg_production.csv")
state_lookup = pd.read_csv("state_lookup.csv")


In [3]:
db_path = Path("usda.db")
if db_path.exists():
    db_path.unlink()
    
with sqlite3.connect("usda.db") as conn:
    conn.execute("""
        CREATE TABLE cheese_production (
            Year INTEGER,
            Period TEXT,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Domain TEXT,
            Value INTEGER
        );
    """)
    conn.execute("""
        CREATE TABLE honey_production (
            Year INTEGER,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Value INTEGER
        );
    """)
    conn.execute("""
        CREATE TABLE milk_production (
            Year INTEGER,
            Period TEXT,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Domain TEXT,
            Value INTEGER
        );
    """)
    conn.execute("""    
        CREATE TABLE coffee_production (
            Year INTEGER,
            Period TEXT,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Value INTEGER
        );
    """)
    conn.execute("""    
        CREATE TABLE egg_production (
            Year INTEGER,
            Period TEXT,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Value INTEGER
        );
    """)
    conn.execute("""
        CREATE TABLE state_lookup (
            State TEXT,
            State_ANSI INTEGER
        );
    """)
    conn.execute("""    
        CREATE TABLE yogurt_production (
            Year INTEGER,
            Period TEXT,
            Geo_Level TEXT,
            State_ANSI INTEGER,
            Commodity_ID INTEGER,
            Domain TEXT,
            Value INTEGER
        );
    """)



In [4]:
integer_columns = {
    "cheese_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
    "honey_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
    "milk_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
    "coffee_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
    "egg_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
    "state_lookup": ["State_ANSI"],
    "yogurt_production": ["Year", "State_ANSI", "Commodity_ID", "Value"],
}

with sqlite3.connect("usda.db") as conn:
    for table, columns in integer_columns.items():
        df = pd.read_csv(f"{table}.csv")

        for column in columns:
            if column in df.columns:
                values = df[column].astype("string").str.replace(",", "", regex=False).str.strip()
                values = values.replace("", pd.NA)
                df[column] = pd.to_numeric(values, errors="coerce").astype("Int64")
                
        df.to_sql(table, conn, if_exists="append", index=False)

    result = conn.execute("""
                SELECT * 
                FROM honey_production
            """).fetchall()
    print(len(result))         

1559


In [5]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT SUM(milk_production.value)
                FROM milk_production
                WHERE milk_production.year = 2023 
            """, conn)
    print(df.to_string(index=False))     

 SUM(milk_production.value)
                91812000000


In [7]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM coffee_production
                WHERE coffee_production.year = 2015 
            """, conn)
    print(df.to_string(index=False))     

 Year Period Geo_Level  State_ANSI  Commodity_ID   Value
 2015   YEAR     STATE          15             1 6600000


In [8]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT AVG(honey_production.value)
                FROM honey_production
                WHERE honey_production.year = 2022 
            """, conn)
    print(df.to_string(index=False))   

 AVG(honey_production.value)
                   3133275.0


In [11]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM state_lookup 
                WHERE state = "IOWA"
            """, conn)
    print(df.to_string(index=False))

State  State_ANSI
 IOWA          19


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT MAX(value)
                FROM yogurt_production
                WHERE year = 2022
            """, conn)
    print(df.to_string(index=False))

In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM honey_production
                WHERE honey_production.state_ansi IN (
                    SELECT state_ansi
                    FROM milk_production
                    WHERE milk_production.year = 2022 )
                    AND honey_production.year = 2022
            """, conn)
    print(df.to_string(index=False))

    

 Year Geo_Level  State_ANSI  Commodity_ID    Value
 2022     STATE           1             2   400000
 2022     STATE           4             2  1035000
 2022     STATE           5             2  1120000
 2022     STATE           6             2 11590000
 2022     STATE           8             2  1302000
 2022     STATE          12             2  7350000
 2022     STATE          13             2  3296000
 2022     STATE          16             2  2726000
 2022     STATE          17             2   470000
 2022     STATE          18             2   567000
 2022     STATE          19             2  2448000
 2022     STATE          20             2   372000
 2022     STATE          21             2   252000
 2022     STATE          22             2  2562000
 2022     STATE          23             2   230000
 2022     STATE          26             2  3362000
 2022     STATE          27             2  5202000
 2022     STATE          28             2  1995000
 2022     STATE          29    

In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM yogurt_production
                WHERE yogurt_production.state_ansi IN (
                    SELECT state_ansi
                    FROM cheese_production
                    WHERE cheese_production.year = 2022 )
                    AND yogurt_production.year = 2022
            """, conn)
    print(df.to_string(index=False))
    

 Year Period Geo_Level  State_ANSI  Commodity_ID Domain     Value
 2022   YEAR     STATE           6             4  TOTAL 377839000
 2022   YEAR     STATE          36             4  TOTAL 793256000


 SUM(value)
91812000000


In [29]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT SUM(value)
                FROM milk_production
                WHERE year = 2023
            """, conn)
    print(df.to_string(index=False))

 SUM(value)
91812000000


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM ( cheese_production LEFT JOIN state_lookup ON cheese_production.state_ansi = state_lookup.state_ansi )
                WHERE year = 2023 AND Period = 'APR' AND Value >= 100000000
            """, conn)
    print(df.to_string(index=False))

    

 Year Period Geo_Level  State_ANSI  Commodity_ID Domain     Value      State  State_ANSI
 2023    APR     STATE         6.0             6  TOTAL 208807000 CALIFORNIA         6.0
 2023    APR     STATE         NaN             6  TOTAL 215206000        NaN         NaN
 2023    APR     STATE        55.0             6  TOTAL 289699000  WISCONSIN        55.0


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT SUM(Value)
                FROM coffee_production
                WHERE year = 2011
            """, conn)
    print(df.to_string(index=False))


    

 SUM(Value)
    7600000


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT AVG(Value)
                FROM honey_production
                WHERE year = 2022
            """, conn)
    print(df.to_string(index=False))

    

 AVG(Value)
  3133275.0


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM state_lookup
            """, conn)
    print(df.to_string(index=False))


    

         State  State_ANSI
       ALABAMA           1
        ALASKA           2
       ARIZONA           4
      ARKANSAS           5
    CALIFORNIA           6
      COLORADO           8
   CONNECTICUT           9
      DELAWARE          10
       FLORIDA          12
       GEORGIA          13
        HAWAII          15
         IDAHO          16
      ILLINOIS          17
       INDIANA          18
          IOWA          19
        KANSAS          20
      KENTUCKY          21
     LOUISIANA          22
         MAINE          23
      MARYLAND          24
 MASSACHUSETTS          25
      MICHIGAN          26
     MINNESOTA          27
   MISSISSIPPI          28
      MISSOURI          29
       MONTANA          30
      NEBRASKA          31
        NEVADA          32
 NEW HAMPSHIRE          33
    NEW JERSEY          34
    NEW MEXICO          35
      NEW YORK          36
NORTH CAROLINA          37
  NORTH DAKOTA          38
          OHIO          39
      OKLAHOMA          40
 

In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM state_lookup LEFT JOIN ( SELECT * FROM cheese_production WHERE Year = 2023 and Period = 'APR') cheese ON state_lookup.state_ansi = cheese.state_ansi
                WHERE State = 'NEW JERSEY'
            """, conn)
    print(df.to_string(index=False))

    

     State  State_ANSI  Year Period Geo_Level  State_ANSI  Commodity_ID Domain   Value
NEW JERSEY          34  2023    APR     STATE          34             6  TOTAL 4889000


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT SUM(yogurt_production.value)
                FROM yogurt_production 
                WHERE yogurt_production.state_ansi IN ( SELECT cheese_production.state_ansi FROM cheese_production WHERE cheese_production.year = 2023 )
                AND yogurt_production.year = 2022
            """, conn)
    print(df.to_string(index=False))


    

 SUM(yogurt_production.value)
                   1171095000


In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM (state_lookup LEFT JOIN ( SELECT * FROM milk_production WHERE Year = 2023 ) milk ON state_lookup.state_ansi = milk.state_ansi) 
                WHERE Value IS NULL
            """, conn)
    print(df.to_string(index=False))
    print(len(df))


    

         State  State_ANSI Year Period Geo_Level State_ANSI Commodity_ID Domain Value
       ALABAMA           1 None   None      None       None         None   None  None
        ALASKA           2 None   None      None       None         None   None  None
      ARKANSAS           5 None   None      None       None         None   None  None
   CONNECTICUT           9 None   None      None       None         None   None  None
      DELAWARE          10 None   None      None       None         None   None  None
        HAWAII          15 None   None      None       None         None   None  None
      KENTUCKY          21 None   None      None       None         None   None  None
     LOUISIANA          22 None   None      None       None         None   None  None
         MAINE          23 None   None      None       None         None   None  None
      MARYLAND          24 None   None      None       None         None   None  None
 MASSACHUSETTS          25 None   None      None      

In [ ]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT *
                FROM (state_lookup LEFT JOIN ( SELECT * FROM cheese_production WHERE Year = 2023 AND Period = 'APR' ) cheese ON state_lookup.state_ansi = cheese.state_ansi) 
            """, conn)
    print(df.to_string(index=False))

    

         State  State_ANSI   Year Period Geo_Level  State_ANSI  Commodity_ID Domain       Value
       ALABAMA           1    NaN    NaN       NaN         NaN           NaN    NaN         NaN
        ALASKA           2    NaN    NaN       NaN         NaN           NaN    NaN         NaN
       ARIZONA           4    NaN    NaN       NaN         NaN           NaN    NaN         NaN
      ARKANSAS           5    NaN    NaN       NaN         NaN           NaN    NaN         NaN
    CALIFORNIA           6 2023.0    APR     STATE         6.0           6.0  TOTAL 208807000.0
      COLORADO           8    NaN    NaN       NaN         NaN           NaN    NaN         NaN
   CONNECTICUT           9    NaN    NaN       NaN         NaN           NaN    NaN         NaN
      DELAWARE          10    NaN    NaN       NaN         NaN           NaN    NaN         NaN
       FLORIDA          12    NaN    NaN       NaN         NaN           NaN    NaN         NaN
       GEORGIA          13    NaN    NaN

In [75]:
with sqlite3.connect("usda.db") as conn:
    df = pd.read_sql_query("""
                SELECT AVG(value)
                FROM coffee_production
                WHERE coffee_production.year IN (SELECT h.year FROM (SELECT SUM(value) as tot, year FROM honey_production GROUP BY year) h WHERE h.tot >= 1000000)
            """, conn)
    print(df.to_string(index=False))

  AVG(value)
6.426667e+06
